# About

This notebook constructs a "validation" set completely outside of the v3v4 selection for the train-test split.

In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from common.preprocess import *

In [2]:
""" dataset directory. """
DATASET_NAME = "validation_set"
NOTEBOOK_CACHE = Path("/data/bwh-comppath-seq/youn/human_microbiome_compendium") / DATASET_NAME
NOTEBOOK_CACHE.mkdir(exist_ok=True, parents=True)

# Dataset Files

File locations and metadata.

In [15]:
data_base_dir = Path("/data/cctm/youn/human_microbiome_compendium")

project_metadata_file = data_base_dir / "projects.csv"
asv_sequence_file = data_base_dir / "obs_md.txt.zst"  # tsv format: ASV_NAME    ASV_SEQ, has a header.
abundance_table_dir = data_base_dir / "asv"
sample_metadata_file = data_base_dir / "sample_metadata.tsv"

assert asv_sequence_file.exists(), f"Expected {asv_sequence_file} to exist."
assert asv_sequence_file.is_file(), f"Expected {asv_sequence_file} to be a file."
assert abundance_table_dir.exists(), f"Expected {asv_sequence_file} to exist."
assert abundance_table_dir.is_dir(), f"Expected {abundance_table_dir} to be a directory."
assert sample_metadata_file.exists(), f"Expected {sample_metadata_file} to exist."
assert sample_metadata_file.is_file(), f"Expected {sample_metadata_file} to be a file."

""" Load the project metadata as a pandas dataframe. """
project_metadata = pd.read_csv(project_metadata_file, sep=',')
print("# projects:", project_metadata.shape[0])

""" Load the sample metadata as a pandas dataframe. """
sample_metadata = pd.read_csv(sample_metadata_file, sep='\t')
print("# samples:", sample_metadata.shape[0])

# projects: 482
# samples: 168464


# Projects -- Target Subset by ID

In [16]:
# PROJECT_IDS = list(project_metadata.loc[project_metadata['amplicon'] == 'v1-v2']['project'])
# print(PROJECT_IDS)

# these are the biggest projects from the above commented-out report!
PROJECT_IDS = ['PRJNA673102', 'PRJDB9469', 'PRJNA682076']

for project_id in PROJECT_IDS:
    print_project_info(project_id, sample_metadata)

Project: PRJNA673102
	regions: ['Europe and Northern America']
	isos: ['DE']
	num samples: 1849
Project: PRJDB9469
	regions: ['Eastern and South-Eastern Asia']
	isos: ['JP']
	num samples: 1070
Project: PRJNA682076
	regions: ['Europe and Northern America']
	isos: ['US']
	num samples: 661


# Sample & project filtering.

In [17]:
project_subset = project_metadata.loc[
    project_metadata['project'].isin(PROJECT_IDS), 
    :
]

sample_subset = sample_metadata.loc[
    (
        sample_metadata['project'].isin(PROJECT_IDS)
    )
]
print("[*] Stage 1 filter: {} samples remaining".format(sample_subset.shape[0]))

[*] Stage 1 filter: 3580 samples remaining


In [18]:
project_subset, sample_subset, asv_seqs_subset, sample_max_num_asvs = filter_samples_and_asvs(
    project_subset, sample_subset,
    abundance_table_dir=abundance_table_dir,
    asv_sequence_file=asv_sequence_file,
    step2_min_num_asv=50,
    step2_max_num_asv=400
)

[Project PRJDB9469] Read count statistics:
         read_counts
count    1070.000000
mean    28485.014019
std     21030.446132
min      5752.000000
25%     14886.750000
50%     20379.000000
75%     35799.000000
max    180649.000000
[Project PRJDB9469] Using read count threshold of 9433.95 < x < 68696.45
*********************************
[Project PRJNA673102] Read count statistics:
         read_counts
count    1849.000000
mean    51500.373175
std     31397.171382
min         1.000000
25%     33124.000000
50%     43709.000000
75%     60611.000000
max    361189.000000
[Project PRJNA673102] Using read count threshold of 22794.8 < x < 100748.79999999996
*********************************
[Project PRJNA682076] Read count statistics:
        read_counts
count  6.610000e+02
mean   2.211762e+05
std    3.019585e+05
min    4.000000e+00
25%    1.735310e+05
50%    2.098530e+05
75%    2.469770e+05
max    6.121885e+06
[Project PRJNA682076] Using read count threshold of 919.0 < x < 309643.0
**********

In [19]:
print("# samples:", sample_subset.shape[0])

# samples: 2353


In [20]:
len(asv_seqs_subset)

78562

# Process 16S sequences.

Keep only sequences that are actually 16S. (some are 18s by accident!)

In [21]:
ASV_SEQ_PROCESSING_DIR = NOTEBOOK_CACHE / "asv_16s_processing"
ASV_SEQ_PROCESSING_DIR.mkdir(exist_ok=True, parents=True)

"""
Note: defer_to_hpc option makes this pipeline print HPC instructions and raise an error.
Follow the directions, and re-run this cell.
"""
asv_seqs_subset = pipeline_16s_validation(
    asv_seqs_subset,
    cache_dir=ASV_SEQ_PROCESSING_DIR,
    silva_db=data_base_dir / "silva_nr99_v138.2_train_set.fa.gz",
    vsearch_path='vsearch',
    vsearch_num_threads=12,
    min_identity=0.95,
)

Wrote 78562 sequences to /data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/asv_sequences.pre_validation.fasta
VSEARCH found: vsearch v2.30.4_linux_x86_64, 62.5GB RAM, 32 cores
Found 78562 sequences in input file.
Using database: /data/cctm/youn/human_microbiome_compendium/silva_nr99_v138.2_train_set.fa.gz
Minimum identity threshold: 95.0%
Strategy: Keeping only bacterial sequences, filtering out Archaea and Eukaryota

Running VSEARCH on /data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/asv_sequences.pre_validation.fasta against /data/cctm/youn/human_microbiome_compendium/silva_nr99_v138.2_train_set.fa.gz...
VSEARCH output /data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/vsearch_results.tsv already exists!
Parsing VSEARCH results and filtering for bacterial sequences...

RESULTS SUMMARY
Total sequences analyzed: 78562
Confirmed bacterial sequences: 71107
Rejected - Ar

In [22]:
""" Run the alignment. """
asv_sequence_file_pre_alignment = ASV_SEQ_PROCESSING_DIR / "asv_sequences.pre_mothur.fasta"
asv_align_file = ASV_SEQ_PROCESSING_DIR / "asv_alignment.fasta"
# REFERENCE_ALIGN_FOR_MOTHUR = Path("/data/cctm/youn/human_microbiome_compendium/GREENGENES_16S_DB") / "gg_13_8_99.refalign"
REFERENCE_ALIGN_FOR_MOTHUR = Path("/data/cctm/youn/human_microbiome_compendium/silva.seed_v138_2/silva.seed_v138_2.align")

dict_to_fasta(asv_seqs_subset, asv_sequence_file_pre_alignment)
bad_asv_ids = run_mothur(
    in_fasta=asv_sequence_file_pre_alignment,
    out_fasta=asv_align_file,
    reference_16s_path=REFERENCE_16S_GREENGENES,
    n_processors=20,
    remove_frac_below=1.0,
    mothur_cmd="mothur"
)

""" Handle the ASVs that mothur suggests removing. """
print("Removing an additional {} ASVs based on MOTHUR output.".format(len(bad_asv_ids)))
for bad_asv_id in bad_asv_ids:
    del asv_seqs_subset[bad_asv_id]

asv_sequence_file_post_filter = ASV_SEQ_PROCESSING_DIR / "asv_sequences.post_filter.fasta"
dict_to_fasta(asv_seqs_subset, asv_sequence_file_post_filter)

Wrote 71107 sequences to /data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/asv_sequences.pre_mothur.fasta
Running mothur command: #align.seqs(fasta=/data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/asv_sequences.pre_mothur.fasta, reference=/data/cctm/youn/human_microbiome_compendium/GREENGENES_16S_DB/gg_13_8_99.refalign, processors=20)
STDOUT:
Linux version

Using ReadLine,Boost,HDF5,GSL
mothur v.1.48.5
Last updated: 1/13/26
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Wrote 538 sequences to /data/bwh-comppath-seq/youn/human_microbiome_compendium/validation_set/asv_16s_processing/asv_sequences.post_filter.fasta


In [23]:
len(bad_asv_ids)

70569

In [24]:
len(asv_seqs_subset)

538